# pcs_citation_trend — patent citations to a paper, by year

`pcs_citation.ipynb` collapses the patent-to-paper citations into fixed windows
(`C_3`, `C_5`, `C_10`, `C_all`). This keeps the **year** instead: one row per
*(paper, citing year)*, which is what a cumulative-trajectory or concavity analysis needs.

    paper_id  pub_year  cite_year  yrs_since_pub  pcs  pcs_examiner  pcs_non_examiner

A citing patent is dated by its **grant** year — when its text became public. Rows where the
citation would precede publication are dropped rather than clamped.

## 1. Setup

In [ ]:
import os, sys, re, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/pcs')
import pcs_common as pcs

# Patent -> paper citations, resolved to the YEAR the citing patent was granted, so a paper's
# technological uptake can be read as a trajectory rather than a single count.
#
#   pcs_citation.ipynb      collapses the same scan into fixed windows (C_3, C_5, C_10, C_all)
#   this notebook           keeps the year, one row per (paper, citing year)
#
# The two are deliberately separate: the windowed file is what most analyses want, and this
# one is ~20x larger because it does not collapse the time axis.
ROOT    = pcs.BASE
PCS     = pcs.PCS_CSV
GPATENT = pcs.granted('g_patent.tsv.zip')
OUT_FP  = pcs.out('pcs_citation_trend.parquet')
NEG     = np.iinfo(np.int32).min
pcs.preflight('pcs_citation')
print(f'\noutput -> {OUT_FP}')

## 2. The two year lookups

In [ ]:
%%time
# citing patent -> grant year
gp = pd.read_csv(GPATENT, sep='\t', usecols=['patent_id', 'patent_date'],
                 dtype={'patent_id': str})
gp['gy'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
pat_year = dict(zip(gp['patent_id'], gp['gy']))
print(f'US patents with a grant year: {len(pat_year):,}')
del gp; gc.collect()

# cited paper -> publication year. oa_common.load_map() is the authoritative per-work table
# and is already sorted by id, so it drops straight into a searchsorted lookup. Unknown years
# are -1 there and become NEG here, because a -1 used as a year would make every lag for
# those papers one year too long instead of dropping them.
uni, _y = pcs.paper_year_map()
pyear = np.where(_y > 0, _y, NEG).astype(np.int64)
print(f'works with a publication year: {int((pyear > NEG).sum()):,} of {len(uni):,}')
del _y; gc.collect()


def paper_year(oaid):
    i = np.clip(np.searchsorted(uni, oaid), 0, len(uni) - 1)
    return np.where(uni[i] == oaid, pyear[i], NEG).astype(np.int64)

## 3. One pass over the citation table

In [ ]:
%%time
# One pass over the 34.8M citation rows, accumulating (paper, citing year, bucket) counts.
# Grouped per chunk so nothing near the full table is ever held at once.
pat_re = re.compile(r'us-0*([0-9]+)-')      # us-000123-b2 -> 123
CAT = {'exm': 'examiner'}                   # everything else is applicant/other
acc = []
t0, seen = time.time(), 0
for ch in pd.read_csv(PCS, usecols=['reftype', 'oaid', 'patent'], dtype=str,
                      chunksize=5_000_000):
    oid = pd.to_numeric(ch['oaid'], errors='coerce')
    pn = ch['patent'].str.lower().str.extract(pat_re, expand=False)
    gy = pn.map(pat_year)
    keep = oid.notna() & gy.notna()
    if keep.any():
        oav = oid.values[keep.values].astype(np.int64)
        py = paper_year(oav)
        ok = py > NEG
        if ok.any():
            d = pd.DataFrame({
                'oaid':      oav[ok],
                'pub_year':  py[ok].astype(np.int32),
                'cite_year': gy.values[keep.values][ok].astype(np.int32),
                'bucket':    ch['reftype'].map(CAT).fillna('non_examiner')
                               .values[keep.values][ok]})
            d = d[d['cite_year'] >= d['pub_year']]     # a citation cannot precede publication
            if len(d):
                acc.append(d.groupby(['oaid', 'pub_year', 'cite_year', 'bucket'])
                            .size().reset_index(name='n'))
    seen += len(ch)
print(f'[{time.time()-t0:.0f}s] scanned {seen:,} PCS rows, '
      f'{sum(len(a) for a in acc):,} partial groups')

## 4. Pivot, derive the lag, write

In [ ]:
%%time
tr = (pd.concat(acc, ignore_index=True)
        .groupby(['oaid', 'pub_year', 'cite_year', 'bucket'])['n'].sum().reset_index())
del acc; gc.collect()
tr = tr.pivot_table(index=['oaid', 'pub_year', 'cite_year'], columns='bucket',
                    values='n', fill_value=0).reset_index()
for b in ('examiner', 'non_examiner'):
    if b not in tr.columns:
        tr[b] = 0
tr = tr.rename(columns={'examiner': 'pcs_examiner', 'non_examiner': 'pcs_non_examiner'})
tr['pcs'] = tr['pcs_examiner'] + tr['pcs_non_examiner']
tr['paper_id'] = 'W' + tr['oaid'].astype('int64').astype(str)
tr['yrs_since_pub'] = (tr['cite_year'] - tr['pub_year']).astype(np.int32)
tr = tr[['paper_id', 'pub_year', 'cite_year', 'yrs_since_pub',
         'pcs', 'pcs_examiner', 'pcs_non_examiner']]
tr.to_parquet(OUT_FP, index=False, compression='zstd')
print(f'WROTE {OUT_FP}  ({len(tr):,} rows, {len(tr.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.1f} MB)')
print(f'  papers {tr.paper_id.nunique():,} | pub_year {tr.pub_year.min()}-{tr.pub_year.max()}'
      f' | cite_year {tr.cite_year.min()}-{tr.cite_year.max()}')
print(f'  yrs_since_pub 0..{tr.yrs_since_pub.max()} | negative lags {int((tr.yrs_since_pub<0).sum())}')
print(f'  citations total {int(tr.pcs.sum()):,} '
      f'(examiner {int(tr.pcs_examiner.sum()):,} / '
      f'non-examiner {int(tr.pcs_non_examiner.sum()):,})')
print('\n  NOTE: the examiner split is near-empty by construction -- this release of Reliance')
print('  on Science tags only 846 of 34.8M rows as `exm`, so pcs == pcs_non_examiner in')
print('  practice. Kept for schema parity with patent_citation_trend, not for analysis.')
display(tr.head(8))